# Native HHL: exact and powered Trotter evolution

Keep the original dense four-by-four matrix, normalized [1,2,4,3] vector, four fractional QPE bits and both evolution routes. Bind classical data before synthesis. The source generic callable is represented by two explicit native root programs; no generic fresh-Output helper or QCallable support is claimed. All numerical results and CX/U3 metrics are computed locally. No provider account is required.


In [ ]:
import math
from pathlib import Path
from typing import Annotated

import torch
from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.algorithms.quantum_linear_solvers.hhl.overview.physics import (
    AMPLITUDES,
    ANGLES,
    MATRIX,
    OPERATOR,
    PAULI_TERMS,
    PAULIS,
    QPE_SIZE,
    REPETITIONS,
    UNITARY,
    B,
)
from examples.synthesis.algorithms.quantum_linear_solvers.hhl.overview.workflow import (
    extract_solution,
    plot_solution,
    verify_program,
)
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QBit,
    QNum,
    allocate,
    apply,
    prepare_amplitudes,
    qfunc,
    qpe,
    select_rotation,
    suzuki_trotter,
    unitary,
    within,
)

A = MATRIX.clone()
b = B.clone()
assert torch.allclose(A, A.mT, rtol=1e-6, atol=1e-6)
eigenvalues = torch.linalg.eigvalsh(A)
assert bool(((eigenvalues > 0) & (eigenvalues < 1)).all())
print("A =", A, "b =", b)
for eigenvalue in eigenvalues:
    binary = format(int(abs(float(eigenvalue)) * 2**32), "032b").rstrip("0")
    print("0." + binary, "=~", float(eigenvalue))
output_directory = Path("artifacts/development/algorithm_hhl")
checks = []
solutions = []
figures = []


In [ ]:
reconstructed = torch.zeros((4, 4), dtype=torch.complex128)
for word, coefficient in PAULI_TERMS:
    reconstructed += coefficient * torch.kron(PAULIS[word[0]], PAULIS[word[1]])
torch.testing.assert_close(reconstructed, A.to(torch.complex128), atol=1e-14, rtol=0)
print("Hamiltonian:", PAULI_TERMS, "Matrix qubits:", 2)


In [ ]:
@qfunc
def load_b(amplitudes: list[float], memory: Output[Annotated[QNum, 2]]) -> None:
    prepare_amplitudes(amplitudes, 0, memory)


## Exact evolution

The native root owns the data, phase and indicator. QPE runs inside within/apply, so the generated inverse acts after reciprocal rotation. Raw phase word k receives rotation 2 asin(1/k), with zero mapped to zero. Finite-QPE residual phase population remains visible and is not called dirty compiler work.


In [ ]:
@qfunc
def exact(
    amplitudes: list[float],
    res: Output[Annotated[QNum, 2]],
    estimator_var: Output[Annotated[QNum, (4, False, 4)]],
    indicator: Output[QBit],
) -> None:
    prepare_amplitudes(amplitudes, 0, res)
    allocate(estimator_var)
    allocate(indicator)
    with within():
        qpe(lambda: unitary(UNITARY, res), estimator_var)
    with apply():
        select_rotation("Y", ANGLES, estimator_var, indicator)


In [ ]:
# The reusable readout helper selects estimator_var=0 and indicator=1,
# retains amplitudes above 1e-12, divides by 1/16, and rounds each complex part
# to five decimals. It reads actual execution amplitudes by named registers.
print("QPE bits:", QPE_SIZE, "Reciprocal scale:", 1 / 2**QPE_SIZE)


In [ ]:
# extract_solution uses a single angle arg(<A q,b>), applies it to the whole
# vector, then takes the real part exactly as the source diagnostic does.
# It also reports complex-state fidelity before that non-unitary projection.
classical = torch.linalg.solve(A, b)
print("Classical solution:", classical)


In [ ]:
amplitudes = list(AMPLITUDES)
assert QPE_SIZE == 4 and len(amplitudes) == 4
print("Exact evolution U = exp(2 pi i A)")


In [ ]:
exact_program = synth.synthesize(
    exact,
    classical_inputs={"amplitudes": amplitudes},
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)


In [ ]:
exact_check = verify_program(exact_program, "exact")
checks.append(exact_check)
print("CX/U3 depth:", exact_check.resources.depth)
print("CX count:", exact_check.resources.operation_counts["cx"])
print(
    "Absolute native and converted errors:",
    exact_check.maximum_state_error,
    exact_check.converted_state_error,
)


In [ ]:
exact_result = fq.run(exact_program)
print(exact_result.statevector())
print(
    dict(
        exact_program.numeric_probabilities(
            exact_result, registers=("res", "estimator_var", "indicator")
        )
    )
)


In [ ]:
exact_solution = extract_solution(exact_program, check=False)
solutions.append(exact_solution)
print(exact_solution)
path = plot_solution(exact_solution, "exact", output_directory)
figures.append(path)
display(Image(filename=str(path)))


## Powered first-order Suzuki–Trotter evolution

Preserve r(power)=4 ceil(1.8**log2(power)), giving 4, 8, 16 and 24 repetitions for powers 1, 2, 4 and 8. The callback evolves with time -2 pi power, order one and the original ordered Pauli decomposition. It does not repeat a fixed approximate U. The independent Circuit multiplies analytic Pauli exponentials for each power separately.


In [ ]:
repetition_table = [1] * 9
for bit, repetitions in enumerate(REPETITIONS):
    qpe_power = 1 << bit
    repetition_table[qpe_power] = 4 * math.ceil(1.8 ** math.log2(qpe_power))
    assert repetition_table[qpe_power] == repetitions
print("Powers:", [1, 2, 4, 8], "Repetitions:", REPETITIONS)


In [ ]:
@qfunc
def trotter(
    amplitudes: list[float],
    repetitions: list[int],
    res: Output[Annotated[QNum, 2]],
    estimator_var: Output[Annotated[QNum, (4, False, 4)]],
    indicator: Output[QBit],
) -> None:
    prepare_amplitudes(amplitudes, 0, res)
    allocate(estimator_var)
    allocate(indicator)
    with within():
        qpe(
            lambda power: suzuki_trotter(
                OPERATOR, -2 * math.pi * power, 1, repetitions[power], res
            ),
            estimator_var,
        )
    with apply():
        select_rotation("Y", ANGLES, estimator_var, indicator)


In [ ]:
trotter_program = synth.synthesize(
    trotter,
    classical_inputs={"amplitudes": amplitudes, "repetitions": repetition_table},
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
trotter_check = verify_program(trotter_program, "trotter")
checks.append(trotter_check)
print(
    "CX/U3 depth:",
    trotter_check.resources.depth,
    "CX count:",
    trotter_check.resources.operation_counts["cx"],
)
trotter_solution = extract_solution(trotter_program)
solutions.append(trotter_solution)
print(trotter_solution)
assert trotter_solution.relative_distance < 0.1
path = plot_solution(trotter_solution, "trotter", output_directory)
figures.append(path)
display(Image(filename=str(path)))
